# 20 | LangChain Middleware：关键工具调用前先让人点头

Agent 会调用工具，这很好。但不是所有工具都应该让它自己说了算。

查订单、查天气、查知识库，通常可以自动执行；但退款、发邮件、删文件、提交审批，就不能让 Agent 一激动直接动手。

这篇讲 Human-in-the-loop，也就是：**关键工具执行前，先暂停，等人确认。**

本篇会做五件事：

- 用 Pydantic `args_schema` 约束退款工单工具参数
- 配置 `HumanInTheLoopMiddleware`，让高风险工具执行前暂停
- 用 `checkpointer` 保存暂停状态
- 演示 `approve`：同意执行
- 演示 `edit` 和 `reject`：修改后执行，或者直接拒绝

一句话：Agent 不是越自动越好。能自动处理小事，也要知道大事先敲门。

## 一、哪些工具需要人工审批

判断标准很简单：**这个工具一旦执行错了，后果是不是要人来收拾。**

一般可以自动执行的工具：

- 查询订单
- 查询退款规则
- 检索知识库
- 读取公开资料

建议人工审批的工具：

- 创建退款工单
- 真正发起退款
- 发送邮件或短信
- 删除文件或数据
- 提交审批、发布内容、执行 SQL 写操作

这篇用客服退款场景：查询订单可以自动执行，创建退款工单必须先让人点头。

## 二、准备真实模型和结构化工具

这里用真实模型演示。因为 HITL 真正要解决的，就是模型在真实业务里准备动手之前，系统能不能及时把它按住。

真实模型要稳定调用工具，关键不是“祈祷它听话”，而是把工具描述和参数 schema 写清楚。

所以退款工单工具会用 Pydantic 定义 `args_schema`。这样模型更容易知道每个参数该填什么，人类审批时也更容易看懂参数含义。

In [ ]:
import os
from typing import Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

load_dotenv(dotenv_path=".env")

# 本地 Ollama，适合本地开发测试。
model_local = init_chat_model("ollama:qwen2.5:14b")

# 云端模型，工具调用稳定性通常更好。本文后面实际使用它演示 HITL。
model_cloud = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)


In [ ]:
class RefundTicketInput(BaseModel):
    """退款工单工具的入参结构。"""

    order_id: str = Field(description="订单号，例如 O-1001")
    amount: int = Field(description="退款金额，单位为元")
    reason: str = Field(description="创建退款工单的原因")
    priority: Literal["low", "normal", "high"] = Field(
        default="normal",
        description="工单优先级：low 低，normal 普通，high 高",
    )


@tool
def get_order_status(order_id: str) -> dict:
    """查询订单状态、商品和订单金额。"""
    # 查询类工具通常可以自动执行。
    return {"order_id": order_id, "status": "未发货", "amount": 389, "item": "蓝牙耳机"}


@tool(args_schema=RefundTicketInput)
def create_refund_ticket(
    order_id: str,
    amount: int,
    reason: str,
    priority: Literal["low", "normal", "high"] = "normal",
) -> dict:
    """创建退款跟进工单。只有用户明确要求创建退款工单时才调用。"""
    # 这个工具会产生业务动作，所以后面要加人工审批。
    print(
        f"[退款工具] 创建工单：order_id={order_id}, "
        f"amount={amount}, priority={priority}, reason={reason}"
    )
    return {
        "message": "退款跟进工单已按当前审批参数创建。",
        "ticket_id": "TK-0001",
        "order_id": order_id,
        "amount": amount,
        "reason": reason,
        "priority": priority,
    }


## 三、创建带人工审批的 Agent

`HumanInTheLoopMiddleware` 通过 `interrupt_on` 指定哪些工具需要暂停。

这里的规则是：

- `get_order_status`：安全查询，自动放行
- `create_refund_ticket`：会创建工单，需要人工审批

注意：HITL 必须配合 `checkpointer`。因为 Agent 要暂停，暂停后还要能从同一个状态恢复。没有状态保存，就像叫暂停但没人记住暂停在哪一秒。

In [ ]:
checkpointer = InMemorySaver()

system_prompt = (
    "你是一个电商售后 Agent。"
    "如果用户只是询问订单状态或退款规则，可以先调用 get_order_status 查询。"
    "如果用户明确要求创建退款跟进工单，必须调用 create_refund_ticket，不能只用文字承诺。"
    "创建退款工单时，order_id 使用用户给出的订单号；amount 使用订单金额；"
    "reason 简洁说明创建原因；priority 默认 normal。"
    "工具执行后，只说明工单已按审批后的参数创建，不要说重新创建，也不要道歉。"
)

agent = create_agent(
    model=model_cloud,
    tools=[get_order_status, create_refund_ticket],
    checkpointer=checkpointer,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # 查询类工具不需要人工审批。
                "get_order_status": False,
                # 创建退款工单需要审批，并允许 approve / edit / reject。
                "create_refund_ticket": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
            },
            description_prefix="这个工具调用需要人工确认",
        )
    ],
    system_prompt=system_prompt,
)

# thread_id 用来标识这一次可暂停、可恢复的对话线程。
config = {"configurable": {"thread_id": "refund-hitl-demo"}}


## 四、第一次调用：Agent 会暂停

第一次调用时，Agent 走到 `create_refund_ticket` 之前会暂停，不会真的执行退款工单工具。

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "请给订单 O-1001 创建退款跟进工单。"}
        ]
    },
    config=config,
    version="v2",
)

# v2 返回 GraphOutput，interrupts 里是等待人工审批的动作。
print(result.interrupts)


你会看到 interrupt 里包含工具名和参数，大概是：

```text
name: create_refund_ticket
args: {
  'order_id': 'O-1001',
  'amount': 389,
  'reason': '用户申请创建退款跟进工单',
  'priority': 'normal'
}
allowed_decisions: ['approve', 'edit', 'reject']
```

如果模型没有显式传 `priority`，工具会使用 Pydantic schema 里定义的默认值 `normal`。

这一步的重点是：**工具还没有执行。** Agent 只是提出了动作申请，等人审批。

## 五、approve：同意执行

如果人工确认没问题，就用 `approve` 恢复执行。

恢复时要用同一个 `config`，也就是同一个 `thread_id`。这样 Agent 才知道要从刚才暂停的地方继续。

In [ ]:
# 用 Command(resume=...) 把人工审批结果交回给 Agent。
# 这里的 approve 表示：同意执行刚才被拦截的 create_refund_ticket 工具调用。
approved_result = agent.invoke(
    Command(
        resume={
            # decisions 是给 HumanInTheLoopMiddleware 的审批结果列表。
            # 如果一次 interrupt 里有多个工具调用，就需要给多个 decision。
            "decisions": [{"type": "approve"}]
        }
    ),
    # 必须使用同一个 config / thread_id，Agent 才能找到刚才暂停的状态。
    config=config,
    # version="v2" 会返回 GraphOutput，方便读取 value 和 interrupts。
    version="v2",
)

# approve 后工具会真正执行，最终回答在 value["messages"] 的最后一条。
print(approved_result.value["messages"][-1].content)


审批通过后，`create_refund_ticket` 才会真正执行。

这就是 HITL 的核心价值：模型可以提出动作，但关键动作由人最后拍板。

## 六、edit：修改参数后执行

有时不是不同意执行，而是参数要改。

比如 Agent 提出的退款金额是 389，但人工审核后只允许先退 199。这个时候用 `edit`。

In [ ]:
# edit 单独使用一个 thread_id，避免和前面的 approve 示例互相影响。
edit_config = {"configurable": {"thread_id": "refund-hitl-edit-demo"}}

# 先让 Agent 再次暂停到人工审批点。
# 这一步只产生 interrupt，不会真正执行 create_refund_ticket。
agent.invoke(
    {"messages": [{"role": "user", "content": "请给订单 O-1001 创建退款跟进工单。"}]},
    config=edit_config,
    version="v2",
)

edited_result = agent.invoke(
    Command(
        resume={
            # edit 表示：这个工具可以执行，但要换成人工审核后的参数。
            "decisions": [
                {
                    "type": "edit",
                    # edited_action 必须写清楚工具名和新的 args。
                    "edited_action": {
                        "name": "create_refund_ticket",
                        "args": {
                            "order_id": "O-1001",
                            "amount": 199,
                            "reason": "人工审核后先退部分金额",
                            "priority": "high",
                        },
                    },
                }
            ]
        }
    ),
    config=edit_config,
    version="v2",
)

print(edited_result.value["messages"][-1].content)


`edit` 适合小范围修改工具参数。

比如退款金额从 389 改成 199，或者把优先级从 `normal` 改成 `high`。

但不要把 `edit` 当成“重新设计任务”。如果人工把参数改得太离谱，流程反而会变复杂。审批不是二次创作，别现场改剧本。

## 七、reject：拒绝执行

如果这个动作不该执行，就用 `reject`。

比如订单信息不足、用户身份没确认、金额异常，都应该拒绝创建退款工单。

In [ ]:
# reject 也单独使用一个 thread_id，方便观察拒绝后的完整流程。
reject_config = {"configurable": {"thread_id": "refund-hitl-reject-demo"}}

# 同样先让 Agent 暂停到人工审批点。
agent.invoke(
    {"messages": [{"role": "user", "content": "请给订单 O-1001 创建退款跟进工单。"}]},
    config=reject_config,
    version="v2",
)

rejected_result = agent.invoke(
    Command(
        resume={
            # reject 表示：这个工具调用不允许执行。
            # message 会作为反馈交还给 Agent，让它生成后续回复。
            "decisions": [
                {
                    "type": "reject",
                    "message": "人工拒绝：订单信息不足，先不要创建退款工单。",
                }
            ]
        }
    ),
    config=reject_config,
    version="v2",
)

print(rejected_result.value["messages"][-1].content)


`reject` 不会执行工具，而是把拒绝原因作为反馈交给 Agent。

这点很重要：拒绝不是报错，而是告诉 Agent“这个动作不允许，换个安全说法或转人工”。

## 八、为什么必须有 checkpointer

HITL 的关键不是“问人一下”，而是：**Agent 会在中途暂停，等人审批后再继续。**

可以把它理解成这个流程：

```mermaid
flowchart TD
    A[生成工具调用] --> B[Middleware 拦截]
    B --> C[保存状态]
    C --> D[返回 interrupt]
    D --> E[人工决定]
    E --> F[同一 thread_id 恢复]
    F --> G[继续执行]
```

这里最关键的是 `保存状态`。

如果没有 checkpointer，Agent 暂停后就不知道自己刚才走到哪一步了。后面即使人类做出了审批决定，也没法从原来的位置继续。

本文用的是 `InMemorySaver`，适合本地演示。生产环境里应该换成持久化存储，比如 Postgres。否则服务一重启，暂停在哪一步就忘了，像开会开到一半大家都失忆。

## 九、怎么选 approve / edit / reject

可以按这个方式记：

| 决策 | 什么时候用 | 结果 |
|---|---|---|
| `approve` | 工具名和参数都没问题 | 原样执行工具 |
| `edit` | 动作可以执行，但参数要改 | 按修改后的参数执行工具 |
| `reject` | 动作不该执行 | 不执行工具，把原因反馈给 Agent |

如果只记一句话：**Agent 可以建议动作，但关键动作要有人负责。**

下一篇继续看敏感信息和上下文管理：用户把手机号、邮箱、身份证号发进来时，Agent 不能假装没看见。

参考：

- LangChain Human-in-the-loop：https://docs.langchain.com/oss/python/langchain/human-in-the-loop
- LangChain Prebuilt Middleware：https://docs.langchain.com/oss/python/langchain/middleware/built-in